# Vantra Advanced Calculator
### Vantra's personal advanced calculator — scientific, programmer & unit converter

In [19]:
import ipywidgets as widgets
from IPython.display import display, HTML
import math
import cmath
import re
from collections import deque

# ─────────────────────────────────────────────
#  Global state
# ─────────────────────────────────────────────
history        = deque(maxlen=50)
memory         = 0.0
current_expr   = ""
result_shown   = False
angle_mode     = "DEG"   # DEG | RAD | GRAD
active_tab     = "standard"  # standard | scientific | programmer | converter

# ─────────────────────────────────────────────
#  Inject global CSS
# ─────────────────────────────────────────────
CSS = """
<style>
/* ── Ocean-blue palette ─────────────────── */
:root {
  --ocean-deep:   #0a2342;
  --ocean-mid:    #1b4f72;
  --ocean-bright: #2e86c1;
  --ocean-light:  #5dade2;
  --ocean-pale:   #aed6f1;
  --foam:         #d6eaf8;
  --white:        #f0f8ff;
  --accent:       #00bcd4;
  --equals:       #00acc1;
  --danger:       #e74c3c;
  --text-dark:    #0a2342;
  --shadow:       rgba(10,35,66,.35);
}

/* ── Wrapper ─────────────────────────────── */
.calc-wrapper {
  font-family: 'Segoe UI', system-ui, sans-serif;
  background: linear-gradient(135deg, #0a2342 0%, #1b4f72 40%, #2e86c1 75%, #5dade2 100%);
  border-radius: 24px;
  padding: 24px;
  max-width: 480px;
  margin: 10px auto;
  box-shadow: 0 20px 60px rgba(0,0,0,.55), inset 0 1px 0 rgba(255,255,255,.15);
}

/* ── Title bar ───────────────────────────── */
.calc-title {
  text-align:center; color:#aed6f1;
  font-size:11px; letter-spacing:3px; text-transform:uppercase;
  margin-bottom:12px; opacity:.8;
}

/* ── Display ─────────────────────────────── */
.calc-display {
  background: linear-gradient(160deg, #051226, #0d2840);
  border-radius: 16px;
  padding: 16px 20px 12px;
  margin-bottom: 14px;
  border: 1px solid rgba(93,173,226,.3);
  box-shadow: inset 0 4px 12px rgba(0,0,0,.4);
}
.display-meta {
  display:flex; justify-content:space-between;
  font-size:10px; color:#5dade2; opacity:.75;
  margin-bottom:6px; letter-spacing:1px;
}
.display-expr {
  min-height:22px; font-size:13px;
  color:#aed6f1; text-align:right;
  word-break:break-all; opacity:.8;
}
.display-main {
  font-size:36px; font-weight:300;
  color:#e8f5ff; text-align:right;
  letter-spacing:-1px; line-height:1.1;
  word-break:break-all;
  text-shadow: 0 0 20px rgba(93,173,226,.5);
}
.display-hist {
  font-size:10px; color:#5dade2;
  text-align:right; margin-top:4px;
  min-height:14px; opacity:.65;
}

/* ── Tab bar ─────────────────────────────── */
.tab-bar {
  display:flex; gap:4px; margin-bottom:14px;
}
.tab-btn {
  flex:1; padding:7px 0; border:none; border-radius:10px;
  font-size:10px; font-weight:600; letter-spacing:.5px;
  cursor:pointer; transition: all .2s;
  background: rgba(255,255,255,.08); color:#aed6f1;
}
.tab-btn:hover  { background:rgba(255,255,255,.16); color:#fff; }
.tab-btn.active { background:linear-gradient(135deg,#00bcd4,#2e86c1); color:#fff;
                  box-shadow:0 4px 12px rgba(0,188,212,.4); }

/* ── Button grid ─────────────────────────── */
.btn-grid {
  display: grid;
  gap: 8px;
}
.btn-grid-4 { grid-template-columns: repeat(4, 1fr); }
.btn-grid-5 { grid-template-columns: repeat(5, 1fr); }

/* ── Buttons ─────────────────────────────── */
.calc-btn {
  padding: 16px 6px;
  border: none; border-radius: 12px;
  font-size: 15px; font-weight: 500;
  cursor: pointer;
  transition: all .15s;
  position: relative; overflow: hidden;
  letter-spacing: .2px;
}
.calc-btn::after {
  content:''; position:absolute; inset:0;
  background:rgba(255,255,255,0); transition:.15s;
  border-radius:inherit;
}
.calc-btn:hover::after  { background:rgba(255,255,255,.12); }
.calc-btn:active::after { background:rgba(255,255,255,.22); }
.calc-btn:active { transform:scale(.95); }

/* colour variants */
.btn-num    { background:linear-gradient(145deg,#1b4f72,#154360);
              color:#e8f5ff; box-shadow:0 3px 8px rgba(0,0,0,.3); }
.btn-op     { background:linear-gradient(145deg,#2e86c1,#1f618d);
              color:#fff;    box-shadow:0 3px 8px rgba(46,134,193,.3); }
.btn-fn     { background:linear-gradient(145deg,#1a5276,#0e3460);
              color:#aed6f1; font-size:12px;
              box-shadow:0 3px 8px rgba(0,0,0,.3); }
.btn-eq     { background:linear-gradient(145deg,#00bcd4,#0097a7);
              color:#fff; font-size:20px;
              box-shadow:0 4px 14px rgba(0,188,212,.45); }
.btn-clear  { background:linear-gradient(145deg,#e74c3c,#c0392b);
              color:#fff; box-shadow:0 3px 8px rgba(231,76,60,.35); }
.btn-mem    { background:linear-gradient(145deg,#0d6e6e,#084f4f);
              color:#7fffd4; font-size:12px;
              box-shadow:0 3px 8px rgba(0,0,0,.3); }
.btn-mode   { background:linear-gradient(145deg,#17202a,#0b1015);
              color:#5dade2; font-size:11px; letter-spacing:.5px;
              box-shadow:0 3px 8px rgba(0,0,0,.4); }
.btn-span2  { grid-column: span 2; }

/* ── History panel ───────────────────────── */
.hist-panel {
  background:rgba(5,18,38,.7); border-radius:14px;
  padding:12px; max-height:160px; overflow-y:auto;
  margin-top:12px;
  border:1px solid rgba(93,173,226,.2);
}
.hist-panel::-webkit-scrollbar { width:4px; }
.hist-panel::-webkit-scrollbar-thumb { background:#2e86c1; border-radius:2px; }
.hist-row {
  display:flex; justify-content:space-between;
  padding:5px 0; border-bottom:1px solid rgba(93,173,226,.1);
  font-size:12px; cursor:pointer;
}
.hist-row:hover { background:rgba(93,173,226,.1); border-radius:6px; padding:5px 6px; }
.hist-expr { color:#7fb3d3; }
.hist-val  { color:#aed6f1; font-weight:600; }
.hist-empty { color:#3d6680; font-size:11px; text-align:center; padding:10px; }

/* ── Converter ───────────────────────────── */
.conv-row { display:flex; gap:8px; align-items:center; margin-bottom:10px; }
.conv-label { color:#aed6f1; font-size:12px; width:60px; }
.conv-result { color:#00bcd4; font-size:18px; font-weight:600;
                background:rgba(5,18,38,.6); border-radius:10px;
                padding:10px 14px; text-align:right; flex:1;
                border:1px solid rgba(0,188,212,.2); }

/* ── Programmer display ──────────────────── */
.prog-row { display:flex; justify-content:space-between;
             padding:5px 0; border-bottom:1px solid rgba(93,173,226,.1); }
.prog-base { color:#5dade2; font-size:11px; width:36px; }
.prog-val  { color:#e8f5ff; font-size:13px; font-family:monospace;
               word-break:break-all; text-align:right; }

.section-label {
  color:#5dade2; font-size:10px; letter-spacing:2px;
  text-transform:uppercase; margin:10px 0 6px;
  opacity:.8;
}

/* ── Angle-mode badge ────────────────────── */
.angle-badge {
  display:inline-block; background:rgba(0,188,212,.2);
  color:#00bcd4; font-size:10px; border-radius:6px;
  padding:2px 8px; margin-left:6px;
  border:1px solid rgba(0,188,212,.4);
}
</style>
"""
display(HTML(CSS))
print("✅  Styles loaded")

✅  Styles loaded


In [20]:
# ─────────────────────────────────────────────
#  Core calculation engine
# ─────────────────────────────────────────────

def to_radians(x):
    """Convert angle to radians based on current mode."""
    if   angle_mode == "DEG":  return math.radians(x)
    elif angle_mode == "GRAD": return x * math.pi / 200
    return x

def from_radians(x):
    if   angle_mode == "DEG":  return math.degrees(x)
    elif angle_mode == "GRAD": return x * 200 / math.pi
    return x

def safe_eval(expr: str):
    """Evaluate a mathematical expression safely."""
    expr = expr.strip()
    if not expr:
        return None, "Empty expression"

    # ── pre-processing ──
    # implicit multiplication  2(3) → 2*(3)  |  2π → 2*π
    expr = re.sub(r'(\d)(\()', r'\1*(\2', expr)
    expr = re.sub(r'(\))(\d)', r'\1*\2', expr)

    # Replace display tokens with python equivalents
    replacements = [
        ("×", "*"), ("÷", "/"), ("−", "-"),
        ("π", str(math.pi)), ("e",  str(math.e)),
        ("**", "**"),
        ("^", "**"),
        ("mod", "%"),
        ("√(", "_sqrt("),
        ("∛(", "_cbrt("),
    ]
    for old, new in replacements:
        expr = expr.replace(old, new)

    # ── build safe namespace ──
    safe_ns = {
        "__builtins__": {},
        "abs": abs, "round": round, "pow": pow,
        "int": int,  "float": float,
        # trig (angle-aware)
        "sin":  lambda x: math.sin(to_radians(x)),
        "cos":  lambda x: math.cos(to_radians(x)),
        "tan":  lambda x: math.tan(to_radians(x)),
        "asin": lambda x: from_radians(math.asin(x)),
        "acos": lambda x: from_radians(math.acos(x)),
        "atan": lambda x: from_radians(math.atan(x)),
        # hyperbolic
        "sinh":  math.sinh,  "cosh":  math.cosh,  "tanh":  math.tanh,
        "asinh": math.asinh, "acosh": math.acosh, "atanh": math.atanh,
        # log / exp
        "log":   math.log10,
        "ln":    math.log,
        "log2":  math.log2,
        "exp":   math.exp,
        # helpers
        "sqrt":  math.sqrt,
        "_sqrt": math.sqrt,
        "_cbrt": lambda x: math.copysign(abs(x)**(1/3), x),
        "cbrt":  lambda x: math.copysign(abs(x)**(1/3), x),
        "floor": math.floor, "ceil": math.ceil,
        "factorial": math.factorial,
        "gcd":   math.gcd,
        "pi":    math.pi,
        "e":     math.e,
        "tau":   math.tau,
        "inf":   math.inf,
    }

    try:
        result = eval(expr, safe_ns)
        if isinstance(result, complex):
            return result, None
        result = float(result)
        if math.isnan(result):  return None, "Not a number"
        if math.isinf(result):  return None, "Infinity"
        return result, None
    except ZeroDivisionError:
        return None, "Division by zero"
    except OverflowError:
        return None, "Overflow"
    except Exception as exc:
        return None, str(exc)


def fmt_number(n):
    """Format a number for display."""
    if isinstance(n, complex):
        r = fmt_number(n.real)
        i = fmt_number(abs(n.imag))
        sign = '+' if n.imag >= 0 else '−'
        return f"{r} {sign} {i}i"
    if n == int(n) and abs(n) < 1e15:
        return f"{int(n):,}"
    if abs(n) >= 1e15 or (abs(n) < 1e-6 and n != 0):
        return f"{n:.6e}"
    # strip trailing zeros
    s = f"{n:.10f}".rstrip('0').rstrip('.')
    # add thousands separator to integer part
    parts = s.split('.')
    parts[0] = f"{int(parts[0]):,}"
    return '.'.join(parts)


print("✅  Engine loaded")

✅  Engine loaded


In [21]:
# ─────────────────────────────────────────────
#  Unit converter data
# ─────────────────────────────────────────────
UNIT_CATEGORIES = {
    "Length": {
        "Meter": 1, "Kilometer": 1e3, "Centimeter": 0.01,
        "Millimeter": 0.001, "Micrometer": 1e-6, "Nanometer": 1e-9,
        "Mile": 1609.344, "Yard": 0.9144, "Foot": 0.3048,
        "Inch": 0.0254, "Nautical Mile": 1852, "Light-year": 9.461e15,
    },
    "Weight": {
        "Kilogram": 1, "Gram": 0.001, "Milligram": 1e-6,
        "Metric Ton": 1000, "Pound": 0.453592, "Ounce": 0.0283495,
        "Stone": 6.35029, "Carat": 0.0002,
    },
    "Temperature": {
        "Celsius": "C", "Fahrenheit": "F", "Kelvin": "K",
        "Rankine": "R",
    },
    "Area": {
        "Sq. Meter": 1, "Sq. Kilometer": 1e6, "Sq. Centimeter": 0.0001,
        "Sq. Mile": 2589988.11, "Sq. Yard": 0.836127, "Sq. Foot": 0.092903,
        "Sq. Inch": 0.00064516, "Hectare": 10000, "Acre": 4046.856,
    },
    "Volume": {
        "Liter": 1, "Milliliter": 0.001, "Cubic Meter": 1000,
        "Cubic Centimeter": 0.001, "Gallon (US)": 3.78541,
        "Quart (US)": 0.946353, "Pint (US)": 0.473176,
        "Cup (US)": 0.236588, "Fluid Oz (US)": 0.0295735,
        "Tablespoon": 0.0147868, "Teaspoon": 0.00492892,
    },
    "Speed": {
        "m/s": 1, "km/h": 0.277778, "mph": 0.44704,
        "knot": 0.514444, "ft/s": 0.3048,
    },
    "Time": {
        "Second": 1, "Minute": 60, "Hour": 3600,
        "Day": 86400, "Week": 604800, "Month": 2628000,
        "Year": 31536000, "Millisecond": 0.001, "Microsecond": 1e-6,
    },
    "Digital": {
        "Bit": 1, "Byte": 8, "Kilobyte": 8192, "Megabyte": 8388608,
        "Gigabyte": 8589934592, "Terabyte": 8796093022208,
        "Kilobit": 1000, "Megabit": 1e6, "Gigabit": 1e9,
    },
    "Angle": {
        "Degree": 1, "Radian": 57.2958, "Gradian": 0.9,
        "Arcminute": 1/60, "Arcsecond": 1/3600,
    },
    "Energy": {
        "Joule": 1, "Kilojoule": 1000, "Calorie": 4.184,
        "Kilocalorie": 4184, "Wh": 3600, "kWh": 3600000,
        "BTU": 1055.06, "eV": 1.60218e-19,
    },
}

def convert_temperature(value, from_unit, to_unit):
    # to Celsius first
    if   from_unit == "Celsius":     c = value
    elif from_unit == "Fahrenheit":  c = (value - 32) * 5/9
    elif from_unit == "Kelvin":      c = value - 273.15
    elif from_unit == "Rankine":     c = (value - 491.67) * 5/9
    else: raise ValueError(from_unit)
    # from Celsius
    if   to_unit == "Celsius":     return c
    elif to_unit == "Fahrenheit":  return c * 9/5 + 32
    elif to_unit == "Kelvin":      return c + 273.15
    elif to_unit == "Rankine":     return (c + 273.15) * 9/5
    else: raise ValueError(to_unit)

def convert_unit(value, from_unit, to_unit, category):
    if category == "Temperature":
        return convert_temperature(value, from_unit, to_unit)
    table  = UNIT_CATEGORIES[category]
    base   = value * table[from_unit]
    result = base  / table[to_unit]
    return result

print("✅  Unit converter loaded")

✅  Unit converter loaded


In [22]:
# ─────────────────────────────────────────────
#  Build the full widget-based UI
# ─────────────────────────────────────────────
from IPython.display import clear_output

# ── output areas ─────────────────────────────
out_main = widgets.Output()

# ── state refs ───────────────────────────────
state = {
    "expr":        "",
    "display":     "0",
    "prev":        "",
    "result_shown": False,
    "angle_mode":  "DEG",
    "tab":         "standard",
    "memory":      0.0,
    "history":     [],
    # converter
    "conv_cat":    "Length",
    "conv_from":   "Meter",
    "conv_to":     "Kilometer",
    "conv_val":    "",
    # programmer
    "prog_val":    0,
}

def render():
    """Re-render the entire calculator into out_main."""
    tab       = state["tab"]
    expr      = state["expr"]
    disp      = state["display"]
    prev      = state["prev"]
    am        = state["angle_mode"]
    mem_val   = state["memory"]
    hist      = state["history"]

    # ── assemble HTML pieces ──

    # display section
    hist_last = f"Ans: {fmt_number(hist[-1][1])}" if hist else ""
    mem_badge = f'<span class="angle-badge">M={fmt_number(mem_val)}</span>' if mem_val != 0 else ''
    angle_badge = f'<span class="angle-badge">{am}</span>'

    display_html = f"""
    <div class="calc-display">
      <div class="display-meta">
        <span> Vantra CALC {angle_badge}{mem_badge}</span>
        <span>{tab.upper()}</span>
      </div>
      <div class="display-expr">{expr or '&nbsp;'}</div>
      <div class="display-main">{disp}</div>
      <div class="display-hist">{hist_last}</div>
    </div>
    """

    # tab bar
    tabs_html = '<div class="tab-bar">'
    for t in ["standard", "scientific", "programmer", "converter"]:
        active = 'active' if t == tab else ''
        label  = t[:3].upper() if t != 'converter' else 'CONV'
        tabs_html += f'<button class="tab-btn {active}" onclick="sendPrompt(\'TAB:{t}\')">{label}</button>'
    tabs_html += '</div>'

    # button grids per tab
    if tab == "standard":
        grid_html = make_standard_grid()
    elif tab == "scientific":
        grid_html = make_scientific_grid()
    elif tab == "programmer":
        grid_html = make_programmer_grid()
    else:
        grid_html = make_converter_grid()

    # history panel
    hist_html = '<div class="hist-panel">'
    if not hist:
        hist_html += '<div class="hist-empty">No history yet</div>'
    else:
        for expr_h, val_h in reversed(hist[-20:]):
            hist_html += (
                f'<div class="hist-row" onclick="sendPrompt(\'HIST:{val_h}\')">'  
                f'<span class="hist-expr">{expr_h}</span>'
                f'<span class="hist-val">{fmt_number(val_h)}</span>'
                f'</div>'
            )
    hist_html += '</div>'

    full_html = f"""
    <div class="calc-wrapper">
      <div class="calc-title">✦ Vantra Advanced Calculator ✦</div>
      {display_html}
      {tabs_html}
      {grid_html}
      <div class="section-label" style="margin-top:14px;">📋 History (click to reuse)</div>
      {hist_html}
    </div>
    """

    with out_main:
        clear_output(wait=True)
        display(HTML(full_html))


# ── button grid builders ──────────────────────

def btn(label, cls, action, span=""):
    span_class = 'btn-span2' if span else ''
    return (f'<button class="calc-btn {cls} {span_class}" '
            f'onclick="sendPrompt(\'BTN:{action}\')">{label}</button>')


def make_standard_grid():
    rows = [
        # row 0 – memory
        btn('MC', 'btn-mem', 'MC') + btn('MR', 'btn-mem', 'MR') +
        btn('M+', 'btn-mem', 'M+') + btn('M−', 'btn-mem', 'M-'),
        # row 1
        btn('C',  'btn-clear', 'C') + btn('⌫', 'btn-op', 'DEL') +
        btn('%',  'btn-op',  '%') + btn('÷', 'btn-op', '÷'),
        # row 2
        btn('7', 'btn-num', '7') + btn('8', 'btn-num', '8') +
        btn('9', 'btn-num', '9') + btn('×', 'btn-op', '×'),
        # row 3
        btn('4', 'btn-num', '4') + btn('5', 'btn-num', '5') +
        btn('6', 'btn-num', '6') + btn('−', 'btn-op', '−'),
        # row 4
        btn('1', 'btn-num', '1') + btn('2', 'btn-num', '2') +
        btn('3', 'btn-num', '3') + btn('+', 'btn-op', '+'),
        # row 5
        btn('±',  'btn-fn',  'NEG') + btn('0', 'btn-num', '0') +
        btn('.',  'btn-num', '.') +   btn('=', 'btn-eq', '='),
        # row 6 – extras
        btn('x²', 'btn-fn', '**2') + btn('x³', 'btn-fn', '**3') +
        btn('√',  'btn-fn', 'SQRT') + btn('1/x', 'btn-fn', '1/x'),
        # row 7
        btn('(',  'btn-fn', '(') + btn(')',  'btn-fn', ')') +
        btn('π',  'btn-fn', 'π') + btn('!',  'btn-fn', '!'),
    ]
    grid = '<div class="btn-grid btn-grid-4">'
    for r in rows:
        grid += r
    grid += '</div>'
    return grid


def make_scientific_grid():
    # row 0 – angle mode
    am = state["angle_mode"]
    angle_row = (
        btn(f'DEG{" ✓" if am=="DEG" else ""}',  'btn-mode', 'MODE:DEG') +
        btn(f'RAD{" ✓" if am=="RAD" else ""}',  'btn-mode', 'MODE:RAD') +
        btn(f'GRAD{" ✓" if am=="GRAD" else ""}', 'btn-mode', 'MODE:GRAD') +
        btn('INV', 'btn-mode', 'INV') + btn('HYP', 'btn-mode', 'HYP')
    )
    inv = state.get("inv", False)
    hyp = state.get("hyp", False)

    sin_lbl  = 'asinh' if (inv and hyp) else ('sinh' if hyp else ('asin' if inv else 'sin'))
    cos_lbl  = 'acosh' if (inv and hyp) else ('cosh' if hyp else ('acos' if inv else 'cos'))
    tan_lbl  = 'atanh' if (inv and hyp) else ('tanh' if hyp else ('atan' if inv else 'tan'))

    rows_inner = [
        btn(sin_lbl,'btn-fn',f'FN:{sin_lbl}(') + btn(cos_lbl,'btn-fn',f'FN:{cos_lbl}(') +
        btn(tan_lbl,'btn-fn',f'FN:{tan_lbl}(') + btn('log','btn-fn','FN:log(') +
        btn('ln','btn-fn','FN:ln('),

        btn('x^y','btn-fn','**') + btn('ⁿ√','btn-fn','FN:NROOT') +
        btn('√','btn-fn','SQRT') + btn('∛','btn-fn','FN:cbrt(') +
        btn('|x|','btn-fn','FN:abs('),

        btn('π','btn-fn','π') + btn('e','btn-fn','e') +
        btn('τ','btn-fn','τ') + btn('mod','btn-fn','FN:mod') +
        btn('n!','btn-fn','FN:!'),

        btn('C','btn-clear','C') + btn('⌫','btn-op','DEL') +
        btn('(','btn-fn','(') + btn(')','btn-fn',')') +
        btn('÷','btn-op','÷'),

        btn('7','btn-num','7') + btn('8','btn-num','8') +
        btn('9','btn-num','9') + btn('×','btn-op','×') +
        btn('%','btn-op','%'),

        btn('4','btn-num','4') + btn('5','btn-num','5') +
        btn('6','btn-num','6') + btn('−','btn-op','−') +
        btn('EE','btn-fn','FN:EE'),

        btn('1','btn-num','1') + btn('2','btn-num','2') +
        btn('3','btn-num','3') + btn('+','btn-op','+') +
        btn('Ans','btn-fn','FN:ANS'),

        btn('±','btn-fn','NEG') + btn('0','btn-num','0') +
        btn('.','btn-num','.') + btn('log₂','btn-fn','FN:log2(') +
        btn('=','btn-eq','='),
    ]
    grid = '<div class="btn-grid btn-grid-5">' + angle_row
    for r in rows_inner:
        grid += r
    grid += '</div>'
    return grid


def make_programmer_grid():
    try:
        iv = int(float(state["display"].replace(',', '')))
    except:
        iv = 0
    state["prog_val"] = iv

    prog_info = f"""
    <div style="background:rgba(5,18,38,.7);border-radius:12px;padding:12px;margin-bottom:10px;
                border:1px solid rgba(93,173,226,.2);">
      <div class="prog-row"><span class="prog-base">DEC</span>
        <span class="prog-val">{iv:,}</span></div>
      <div class="prog-row"><span class="prog-base">HEX</span>
        <span class="prog-val">{hex(iv).upper()[2:] if iv >= 0 else '-'+hex(-iv).upper()[2:]}</span></div>
      <div class="prog-row"><span class="prog-base">BIN</span>
        <span class="prog-val" style="font-size:11px">{bin(iv)[2:] if iv >= 0 else '-'+bin(-iv)[2:]}</span></div>
      <div class="prog-row" style="border-bottom:none"><span class="prog-base">OCT</span>
        <span class="prog-val">{oct(iv)[2:] if iv >= 0 else '-'+oct(-iv)[2:]}</span></div>
    </div>
    """

    btns = [
        btn('C','btn-clear','C') + btn('⌫','btn-op','DEL') +
        btn('÷','btn-op','÷') + btn('×','btn-op','×'),

        btn('AND','btn-fn','FN: and ') + btn('OR','btn-fn','FN: or ') +
        btn('XOR','btn-fn','FN: ^ ') + btn('NOT','btn-fn','FN:~'),

        btn('<<','btn-fn','FN:<<') + btn('>>','btn-fn','FN:>>') +
        btn('MOD','btn-fn','%') + btn('(','btn-fn','('),

        btn('F','btn-fn','FN:0xF') + btn('E','btn-fn','FN:0xE') +
        btn('D','btn-fn','FN:0xD') + btn(')','btn-fn',')'),

        btn('B','btn-fn','FN:0xB') + btn('A','btn-fn','FN:0xA') +
        btn('−','btn-op','−') + btn('+','btn-op','+'),

        btn('7','btn-num','7') + btn('8','btn-num','8') +
        btn('9','btn-num','9') + btn('0x','btn-fn','FN:0x'),

        btn('4','btn-num','4') + btn('5','btn-num','5') +
        btn('6','btn-num','6') + btn('0b','btn-fn','FN:0b'),

        btn('1','btn-num','1') + btn('2','btn-num','2') +
        btn('3','btn-num','3') + btn('=','btn-eq','='),

        btn('±','btn-fn','NEG') + btn('0','btn-num','0') +
        btn('.','btn-num','.') + btn('','btn-num',''),  # spacer
    ]
    grid = prog_info + '<div class="btn-grid btn-grid-4">'
    for r in btns:
        grid += r
    grid += '</div>'
    return grid


def make_converter_grid():
    cat      = state["conv_cat"]
    from_u   = state["conv_from"]
    to_u     = state["conv_to"]
    val_str  = state["conv_val"]

    # category row
    cats_html = '<div style="display:flex;flex-wrap:wrap;gap:5px;margin-bottom:10px;">'
    for c in UNIT_CATEGORIES:
        active = 'background:linear-gradient(135deg,#00bcd4,#2e86c1);color:#fff;box-shadow:0 3px 8px rgba(0,188,212,.4);' if c == cat else ''
        cats_html += (f'<button style="padding:5px 10px;border:none;border-radius:8px;'
                      f'background:rgba(255,255,255,.1);color:#aed6f1;font-size:11px;cursor:pointer;{active}"'
                      f' onclick="sendPrompt(\'CONV_CAT:{c}\')">{ c}</button>')
    cats_html += '</div>'

    # unit selectors (html selects won't fire easily; use buttons)
    units = list(UNIT_CATEGORIES[cat].keys())

    def unit_row(label, selected, prefix):
        html = f'<div class="section-label">{label}</div><div style="display:flex;flex-wrap:wrap;gap:4px;margin-bottom:8px;">'
        for u in units:
            a = 'background:linear-gradient(135deg,#00bcd4,#2e86c1);color:#fff;' if u == selected else ''
            html += (f'<button style="padding:4px 9px;border:none;border-radius:7px;'
                     f'background:rgba(255,255,255,.08);color:#aed6f1;font-size:11px;cursor:pointer;{a}"'
                     f' onclick="sendPrompt(\'{ prefix}:{u}\')">{u}</button>')
        html += '</div>'
        return html

    from_row = unit_row('FROM', from_u, 'CONV_FROM')
    to_row   = unit_row('TO',   to_u,   'CONV_TO')

    # numeric input
    result_html = ''
    if val_str:
        try:
            v = float(val_str)
            r = convert_unit(v, from_u, to_u, cat)
            result_html = f"""
            <div style="background:rgba(5,18,38,.7);border-radius:12px;padding:14px;
                        border:1px solid rgba(0,188,212,.3);margin-top:10px;">
              <div style="color:#aed6f1;font-size:13px;">
                {fmt_number(v)} <span style="color:#5dade2">{from_u}</span>
              </div>
              <div style="font-size:28px;color:#00bcd4;font-weight:300;margin:6px 0;">
                {fmt_number(r)}
              </div>
              <div style="color:#5dade2;font-size:13px;">{to_u}</div>
            </div>"""
        except Exception as ex:
            result_html = f'<div style="color:#e74c3c;">{ex}</div>'

    # keypad for value entry
    kpad = '<div class="btn-grid btn-grid-4" style="margin-top:10px;">'
    for ch in ['7','8','9','⌫','4','5','6','C','1','2','3','.','±','0','','=']:
        if ch == '':
            kpad += '<div></div>'
        elif ch == '=':
            kpad += btn(ch, 'btn-eq', 'CONV_CALC')
        elif ch == 'C':
            kpad += btn(ch, 'btn-clear', 'CONV_CLR')
        elif ch == '⌫':
            kpad += btn(ch, 'btn-op', 'CONV_DEL')
        else:
            kpad += btn(ch, 'btn-num', f'CONV_IN:{ch}')
    kpad += '</div>'

    conv_display = f"""
    <div style="background:rgba(5,18,38,.7);border-radius:12px;padding:10px 14px;
                border:1px solid rgba(93,173,226,.2);margin-bottom:8px;">
      <div style="color:#5dade2;font-size:10px;letter-spacing:1px;">INPUT VALUE</div>
      <div style="color:#e8f5ff;font-size:24px;text-align:right;">{val_str or '0'}</div>
    </div>"""

    return cats_html + from_row + to_row + conv_display + result_html + kpad

print("✅  UI builders ready")

✅  UI builders ready


In [23]:
# ─────────────────────────────────────────────
#  Event handler  (receives sendPrompt calls)
# ─────────────────────────────────────────────

txt_input = widgets.Text(
    placeholder='(internal command bus – ignore)',
    layout=widgets.Layout(display='none')
)

def handle_action(action: str):
    """Dispatch an action string and re-render."""

    # ── tab switch ────────────────────────────
    if action.startswith('TAB:'):
        state["tab"] = action[4:]
        render(); return

    # ── angle mode ───────────────────────────
    if action.startswith('MODE:'):
        state["angle_mode"] = action[5:]
        render(); return

    # ── INV / HYP toggles ────────────────────
    if action == 'BTN:INV':
        state["inv"] = not state.get("inv", False)
        render(); return
    if action == 'BTN:HYP':
        state["hyp"] = not state.get("hyp", False)
        render(); return

    # ── history recall ───────────────────────
    if action.startswith('HIST:'):
        val = action[5:]
        state["expr"]        = val
        state["display"]     = val
        state["result_shown"] = True
        render(); return

    # ── converter ───────────────────────────
    if action.startswith('CONV_CAT:'):
        state["conv_cat"]  = action[9:]
        units = list(UNIT_CATEGORIES[state["conv_cat"]].keys())
        state["conv_from"] = units[0]
        state["conv_to"]   = units[1] if len(units) > 1 else units[0]
        render(); return
    if action.startswith('CONV_FROM:'):
        state["conv_from"] = action[10:]; render(); return
    if action.startswith('CONV_TO:'):
        state["conv_to"]   = action[8:];  render(); return
    if action.startswith('CONV_IN:'):
        ch = action[8:]
        if ch == '±':
            v = state["conv_val"]
            state["conv_val"] = v[1:] if v.startswith('-') else ('-'+v if v else v)
        elif ch == '.' and '.' not in state["conv_val"]:
            state["conv_val"] += ch
        elif ch not in ('.', '±'):
            state["conv_val"] += ch
        render(); return
    if action == 'CONV_DEL':
        state["conv_val"] = state["conv_val"][:-1]; render(); return
    if action == 'CONV_CLR':
        state["conv_val"] = ""; render(); return
    if action == 'CONV_CALC':
        render(); return   # result shown automatically

    # ── main calculator ──────────────────────
    if not action.startswith('BTN:'):
        render(); return
    key = action[4:]

    expr = state["expr"]
    rs   = state["result_shown"]

    # clear all
    if key == 'C':
        state.update(expr='', display='0', prev='', result_shown=False)
        render(); return

    # delete last char
    if key == 'DEL':
        if rs:
            state.update(expr='', display='0', result_shown=False)
        else:
            state["expr"] = expr[:-1] or ''
            state["display"] = state["expr"] or '0'
        render(); return

    # equals
    if key == '=':
        if not expr: render(); return
        result, err = safe_eval(expr)
        if err:
            state["display"]     = f"Error: {err}"
            state["result_shown"] = False
        else:
            state["prev"]        = expr
            state["display"]     = fmt_number(result)
            state["result_shown"] = True
            state["history"].append((expr, result))
            state["expr"] = str(result) if not isinstance(result, complex) else str(result)
        render(); return

    # memory
    if key == 'MC': state["memory"] = 0.0; render(); return
    if key == 'MR':
        m = fmt_number(state["memory"])
        state["expr"]        = str(state["memory"])
        state["display"]     = m
        state["result_shown"] = True
        render(); return
    if key in ('M+', 'M-'):
        try:
            v = float(state["display"].replace(',', '').replace('Error', '0'))
            state["memory"] += v if key == 'M+' else -v
        except: pass
        render(); return

    # special operations on current result
    if key == 'NEG':
        cur = state["expr"]
        if cur.startswith('-'):
            state["expr"] = cur[1:]
        else:
            state["expr"] = '-' + cur if cur else '-'
        state["display"] = state["expr"]
        render(); return

    if key == 'SQRT':
        state["expr"] += 'sqrt('
        state["display"] = state["expr"]
        state["result_shown"] = False
        render(); return

    if key == '**2':
        state["expr"] += '**2'
        state["display"] = state["expr"]
        render(); return

    if key == '**3':
        state["expr"] += '**3'
        state["display"] = state["expr"]
        render(); return

    if key == '1/x':
        cur = state["expr"]
        state["expr"]    = f'1/({cur})' if cur else '1/'
        state["display"] = state["expr"]
        render(); return

    if key == '!':
        state["expr"] += '!'
        # handle factorial in expression
        # replace n! with factorial(n)
        import re as _re
        e2 = _re.sub(r'(\d+)!', lambda m: f'factorial({m.group(1)})', state["expr"])
        state["expr"] = e2
        state["display"] = state["expr"]
        render(); return

    # function tokens (from scientific)
    if key.startswith('FN:'):
        fn = key[3:]
        if fn == 'ANS':
            if state["history"]:
                ans = str(state["history"][-1][1])
                state["expr"]   += ans
                state["display"] = state["expr"]
            render(); return
        if fn == 'EE':
            state["expr"]   += 'e+'
            state["display"] = state["expr"]
            render(); return
        if fn == '!':
            state["expr"] = f'factorial({state["expr"]})'
            state["display"] = state["expr"]
            render(); return
        if fn == 'NROOT':
            state["expr"] += '**(1/'
            state["display"] = state["expr"]
            render(); return
        if fn == 'mod':
            state["expr"]   += '%'
            state["display"] = state["expr"]
            render(); return
        if fn in ('<<', '>>'):
            state["expr"] += fn
            state["display"] = state["expr"]
            render(); return
        if fn == '~':
            state["expr"] = f'~int({state["expr"]})'
            state["display"] = state["expr"]
            render(); return
        if fn.startswith('0x') or fn.startswith('0b'):
            state["expr"]   += fn
            state["display"] = state["expr"]
            render(); return
        # generic fn: append token
        if rs:
            state["expr"]        = fn
            state["result_shown"] = False
        else:
            state["expr"] += fn
        state["display"] = state["expr"]
        render(); return

    # ── default: append character ────────────
    operators = set('÷×−+%')
    if rs:
        if key in operators or key in ('**', '('):
            # continue with result
            pass
        else:
            # start fresh number
            state["expr"] = ''
        state["result_shown"] = False

    state["expr"]   += key
    state["display"] = state["expr"]
    render()


def on_text_change(change):
    """Called when JS sendPrompt posts a message."""
    val = change['new'].strip()
    if val:
        handle_action(val)
        txt_input.value = ''

txt_input.observe(on_text_change, names='value')

print("✅  Event handler ready")

✅  Event handler ready


In [24]:
# ─────────────────────────────────────────────
#  Launch the calculator
# ─────────────────────────────────────────────

# The sendPrompt JS function routes clicks into the hidden text widget
js_bridge = HTML("""
<script>
function sendPrompt(msg) {
  // find the hidden text input and set its value
  var inputs = document.querySelectorAll('input.widget-text');
  for (var i = 0; i < inputs.length; i++) {
    var inp = inputs[i];
    if (inp.closest('.widget-hidden') || inp.style.display === 'none') continue;
    // find any hidden text widget
  }
  // target ALL text inputs (the hidden one is the first/only one here)
  var allInputs = document.querySelectorAll('input[type="text"]');
  for (var j = 0; j < allInputs.length; j++) {
    var nativeInput = allInputs[j];
    var nativeInputValueSetter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
    nativeInputValueSetter.call(nativeInput, msg);
    nativeInput.dispatchEvent(new Event('input', { bubbles: true }));
    break;
  }
}
</script>
""")

display(js_bridge)
display(txt_input)
display(out_main)

render()
print("\nVantra Advanced Calculator is ready! Interact with the buttons above.")

Text(value='', layout=Layout(display='none'), placeholder='(internal command bus – ignore)')

Output()


Vantra Advanced Calculator is ready! Interact with the buttons above.
